# Checkpoint 5B — Quantitative Magnetic-Coordinate Framing (NOAA-19, January 2024)

**Question:** how can the particle-defined high-flux footprint be *described* in magnetic-coordinate
terms, and which NOAA-provided magnetic variables best separate high-flux samples from the broader
regional sample?

**Still descriptive** — not a causal model, not a classification rule, not a final SAA boundary; the
particle footprint is **not** equated with the magnetic-field minimum. Magnetic variables are
NOAA-provided IGRF model quantities (no external IGRF added). `L_IGRF == -1` is a documented invalid
sentinel, excluded wherever L is used. `mag_lon_sat` uses wrap-aware handling only. No dose / health /
danger / discovery / causal claims.

In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from saa.magnetic_framing import (
    magnetic_validity_table, add_footprint_flags, all_binned_profiles, footprint_magnetic_summary,
    concentration_metrics, save_table, PROFILE_VARIABLES,
    plot_flux_profile, plot_inside_outside, plot_btot_vs_l, plot_high_flux_btot_vs_l,
    plot_mag_lat_lon_wrapaware,
)
PROC = ROOT/"data"/"processed"; TBL = ROOT/"outputs"/"tables"; FIG = ROOT/"outputs"/"figures"
for d in (TBL, FIG): d.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 240, "display.max_columns", 30)

CP5A = PROC / "cp5a_noaa19_2024-01_region_flux_plus_magnetic.parquet"
df = pd.read_parquet(CP5A)
g5 = pd.read_parquet(TBL / "cp4a_noaa19_2024-01_grid_5deg.parquet")
g2 = pd.read_parquet(TBL / "cp4a_noaa19_2024-01_grid_2deg.parquet")
print("CP5A flux+magnetic rows:", len(df), "| columns:", list(df.columns))

CP5A flux+magnetic rows: 205153 | columns: ['time', 'lat', 'lon', 'alt', 'satellite', 'source_file', 'mep_omni_flux_p1', 'mep_IFC_on', 'mep_omni_flux_flag_fit', 'L_IGRF', 'Btot_sat', 'mag_lat_sat', 'mag_lon_sat', 'MLT', 'lon180']


## 1. Magnetic variable validity rules + invalid/excluded counts

In [2]:
validity = magnetic_validity_table(df)
save_table(validity, TBL/"cp5b_magnetic_variable_validity.csv")
print(validity.to_string(index=False))
print("\nL_IGRF invalid (-1 sentinel) excluded:",
      int(validity.loc[validity.variable_name=='L_IGRF','rows_invalid'].iloc[0]), "rows")

variable_name  rows_total  rows_valid  rows_invalid                                                          invalid_rule    valid_min    valid_max                                                                 note
     Btot_sat      205153      205153             0                                                        finite and > 0 16015.219727 32434.179688                                                                     
       L_IGRF      205153      201239          3914                                       finite, != -1 sentinel, and > 0     1.100000     6.480000                                                                     
  mag_lat_sat      205153      205153             0                                       finite and within [-90, 90] deg   -66.559998    31.580000                                                                     
  mag_lon_sat      205153      205153             0 finite and within [0, 360] deg (use circular/wrap-aware methods only)     0.0000

## 2. Assign footprint membership (4 pilot cases) then build magnetic-binned flux profiles

In [3]:
df = add_footprint_flags(df, g5, g2)
print("footprint sample counts:",
      {c:int(df[c].sum()) for c in ['in_top10_5deg','in_top5_5deg','in_top10_2deg','in_top5_2deg']})
profiles = all_binned_profiles(df, PROFILE_VARIABLES)
save_table(profiles, TBL/"cp5b_magnetic_binned_flux_profiles.csv",
           TBL/"cp5b_magnetic_binned_flux_profiles.parquet")
print("\nprofile rows:", len(profiles), "| variables:", list(profiles.variable.unique()))
# show Btot_sat profile (bin edges documented in the table)
b = profiles[profiles.variable=='Btot_sat']
print(b[['bin_left','bin_right','sample_count','median_flux','p90_flux',
         'fraction_of_samples_in_top10_geographic_footprint']].to_string(index=False))

footprint sample counts: {'in_top10_5deg': 20829, 'in_top5_5deg': 10040, 'in_top10_2deg': 20317, 'in_top5_2deg': 10029}



profile rows: 50 | variables: ['Btot_sat', 'L_IGRF', 'mag_lat_sat', 'MLT']
    bin_left    bin_right  sample_count  median_flux  p90_flux  fraction_of_samples_in_top10_geographic_footprint
16015.219727 17188.002581         22380    19.563499 29.893801                                           0.810813
17188.002581 18360.785435         32333     4.896900 12.351700                                           0.082980
18360.785435 19533.568290         37989     0.806600  3.530200                                           0.000000
19533.568290 20706.351144         32132     0.122900  0.804300                                           0.000000
20706.351144 21879.133998         22096     0.018150  0.098300                                           0.000000
21879.133998 23051.916853         16982     0.010000  0.048000                                           0.000000
23051.916853 24224.699707         12334     0.011050  0.048300                                           0.000000
24224.699707

The `bin_left`/`bin_right` columns document every bin edge. Note how the
`fraction_of_samples_in_top10_geographic_footprint` rises sharply toward the **lowest** `Btot_sat`
bins — a descriptive concentration, not a boundary.

## 3. Footprint magnetic summaries (inside vs outside, 4 cases, with separation metric)

In [4]:
summary = footprint_magnetic_summary(df)
save_table(summary, TBL/"cp5b_footprint_magnetic_summary.csv",
           TBL/"cp5b_footprint_magnetic_summary.parquet")
print("summary rows:", len(summary))
print("separation_metric := (median_out - median_in)/(0.5*(iqr_in+iqr_out)); +ve = footprint lower\n")
print(summary[['comparison_case','magnetic_variable','median_inside','median_outside',
               'iqr_inside','iqr_outside','separation_metric']].to_string(index=False))

summary rows: 16
separation_metric := (median_out - median_in)/(0.5*(iqr_in+iqr_out)); +ve = footprint lower

comparison_case magnetic_variable  median_inside  median_outside  iqr_inside  iqr_outside  separation_metric
top10_5deg_mean          Btot_sat   16621.070312    20242.230469  593.019531  3978.401855           1.584260
top10_5deg_mean            L_IGRF       1.290000        1.550000    0.130000     1.130000           0.412698
top10_5deg_mean       mag_lat_sat      -9.550000      -20.820000   10.520000    49.449998          -0.375855
top10_5deg_mean               MLT       8.900000        9.580000   12.050001    12.199999           0.056082
 top5_5deg_mean          Btot_sat   16363.635254    20042.320312  349.807861  4026.990234           1.680994
 top5_5deg_mean            L_IGRF       1.300000        1.500000    0.080000     1.060000           0.350877
 top5_5deg_mean       mag_lat_sat     -10.180000      -18.969999    7.420000    47.130001          -0.322273
 top5_5deg_mean   

## 4. Magnetic-space concentration metrics (Btot_sat, L_IGRF; descriptive only)

In [5]:
conc = concentration_metrics(df)
save_table(conc, TBL/"cp5b_magnetic_concentration_metrics.csv")
print(conc.to_string(index=False))

                              metric footprint variable    value                                                                   definition_note
         fraction_below_regional_q25     top10 Btot_sat 1.000000           frac of top10 footprint samples with Btot_sat < regional q25 (18258 nT)
fraction_in_regional_lowest_quartile     top10   L_IGRF 0.085635                frac of top10 footprint samples with L_IGRF < regional q25 (1.220)
  regional_fraction_to_capture_50pct     top10 Btot_sat 0.050957 smallest low-Btot_sat regional fraction containing 50% of top10 footprint samples
  regional_fraction_to_capture_75pct     top10 Btot_sat 0.083811 smallest low-Btot_sat regional fraction containing 75% of top10 footprint samples
  regional_fraction_to_capture_90pct     top10 Btot_sat 0.120371 smallest low-Btot_sat regional fraction containing 90% of top10 footprint samples
  regional_fraction_to_capture_50pct     top10   L_IGRF 0.367518   smallest low-L_IGRF regional fraction containing 50

## 5. Read the descriptive framing

In [6]:
s10 = summary[summary.comparison_case=='top10_5deg_mean'].set_index('magnetic_variable')
print("== which variable separates the footprint most (top10 5deg mean, |separation_metric|) ==")
for v in ['Btot_sat','L_IGRF','mag_lat_sat','MLT']:
    print(f"  {v:12s} separation_metric = {s10.loc[v,'separation_metric']:+.2f}")
c = conc.set_index(['metric','footprint','variable'])['value']
print("\n== low-Btot concentration ==")
print(f"  {c[('fraction_below_regional_q25','top10','Btot_sat')]*100:.0f}% of top10 footprint samples are below the regional Btot q25")
print(f"  {c[('fraction_below_regional_q25','top5','Btot_sat')]*100:.0f}% of top5 footprint samples are below the regional Btot q25")
print(f"  50% of top10 footprint captured within the lowest {c[('regional_fraction_to_capture_50pct','top10','Btot_sat')]*100:.1f}% of regional Btot")
print(f"  90% of top10 footprint captured within the lowest {c[('regional_fraction_to_capture_90pct','top10','Btot_sat')]*100:.1f}% of regional Btot")
print("\n== low-L concentration (weaker than Btot) ==")
print(f"  only {c[('fraction_in_regional_lowest_quartile','top10','L_IGRF')]*100:.1f}% of top10 footprint is in the regional lowest-L quartile")
print(f"  50% of top10 footprint captured within the lowest {c[('regional_fraction_to_capture_50pct','top10','L_IGRF')]*100:.1f}% of regional L")
print("\nMLT separation is ~0 (local-time diagnostic; non-discrimination is not physical proof).")

== which variable separates the footprint most (top10 5deg mean, |separation_metric|) ==
  Btot_sat     separation_metric = +1.58
  L_IGRF       separation_metric = +0.41
  mag_lat_sat  separation_metric = -0.38
  MLT          separation_metric = +0.06

== low-Btot concentration ==
  100% of top10 footprint samples are below the regional Btot q25
  100% of top5 footprint samples are below the regional Btot q25
  50% of top10 footprint captured within the lowest 5.1% of regional Btot
  90% of top10 footprint captured within the lowest 12.0% of regional Btot

== low-L concentration (weaker than Btot) ==
  only 8.6% of top10 footprint is in the regional lowest-L quartile
  50% of top10 footprint captured within the lowest 36.8% of regional L

MLT separation is ~0 (local-time diagnostic; non-discrimination is not physical proof).


## 6. Diagnostic figures (all DESCRIPTIVE; no boundary lines)

In [7]:
# A flux profiles
plot_flux_profile(profiles, "Btot_sat",    FIG/"cp5b_flux_profile_by_Btot_sat.png")
plot_flux_profile(profiles, "L_IGRF",      FIG/"cp5b_flux_profile_by_L_IGRF.png")
plot_flux_profile(profiles, "mag_lat_sat", FIG/"cp5b_flux_profile_by_mag_lat_sat.png")
plot_flux_profile(profiles, "MLT",         FIG/"cp5b_flux_profile_by_MLT.png")
# B inside/outside
plot_inside_outside(df, "Btot_sat",    FIG/"cp5b_inside_outside_Btot_sat.png")
plot_inside_outside(df, "L_IGRF",      FIG/"cp5b_inside_outside_L_IGRF.png")
plot_inside_outside(df, "mag_lat_sat", FIG/"cp5b_inside_outside_mag_lat_sat.png")
# C Btot vs L
plot_btot_vs_l(df,           FIG/"cp5b_flux_Btot_vs_L_IGRF.png")
plot_high_flux_btot_vs_l(df, FIG/"cp5b_high_flux_footprint_Btot_vs_L_IGRF.png")
# D optional wrap-aware mag lat/lon
plot_mag_lat_lon_wrapaware(df, FIG/"cp5b_mag_lat_vs_mag_lon_wrapaware.png")
print("CP5B figures:", sorted(p.name for p in FIG.glob("cp5b_*.png")))

CP5B figures: ['cp5b_flux_Btot_vs_L_IGRF.png', 'cp5b_flux_profile_by_Btot_sat.png', 'cp5b_flux_profile_by_L_IGRF.png', 'cp5b_flux_profile_by_MLT.png', 'cp5b_flux_profile_by_mag_lat_sat.png', 'cp5b_high_flux_footprint_Btot_vs_L_IGRF.png', 'cp5b_inside_outside_Btot_sat.png', 'cp5b_inside_outside_L_IGRF.png', 'cp5b_inside_outside_mag_lat_sat.png', 'cp5b_mag_lat_vs_mag_lon_wrapaware.png']


## 7. Summary

- **Validity:** `L_IGRF` had 3,914 invalid (-1) rows excluded (valid 1.10–6.48); `Btot_sat`,
  `mag_lat_sat`, `MLT` fully valid; `mag_lon_sat` handled wrap-aware only (no naive IQR/mean).
- **Most structuring variable:** **`Btot_sat`** separates the particle high-flux footprint most
  strongly (separation_metric ≈ 1.6; the footprint sits in a low, narrow field-strength band). `L_IGRF`
  separates moderately; `mag_lat_sat` shifts but is broad; **`MLT` is essentially non-discriminating**.
- **Narrower than geography alone?** Yes, descriptively: ~100% of the top10/top5 footprint samples lie
  below the regional `Btot_sat` 25th percentile, and 50%/90% of the top10 footprint is captured within
  only the lowest ~5%/~12% of regional `Btot_sat` — a tight low-Btot concentration. `L_IGRF`
  concentrates more weakly (footprint median L just above the regional L-q25).
- **Unresolved:** causality (none claimed); foot-point vs satellite-coordinate choice; wrap-aware
  magnetic-longitude *statistics* (only plotted, not summarized); multi-satellite / multi-month
  generality; whether a low-Btot concentration is partly an orbital-sampling effect.

This is **descriptive magnetic-coordinate framing** — *not* a final field-model explanation, true SAA
boundary, dose, health risk, danger zone, or discovery. `mep_IFC_on==-1` retained, uninterpreted.